<a href="https://colab.research.google.com/github/Murcha1990/LLM_Course_2026/blob/main/Lection3_Transformer/LLM_zero_few_shot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
!pip install transformers -q

# Пример LLM

Сегодня мы рассмотрим, как работать с декодерными моделями в режиме zero-shot. Мы начнём с базового примера zero-shot для GPT-3, а затем перейдём к более продвинутым LLM.

In [ ]:
import torch

# If there's a GPU available...
if torch.cuda.is_available():

    # Tell PyTorch to use the GPU.
    device = torch.device("cuda")

    print('There are %d GPU(s) available.' % torch.cuda.device_count())

    print('We will use the GPU:', torch.cuda.get_device_name(0))

# If not...
else:
    print('No GPU available, using the CPU instead.')
    device = torch.device("cpu")
device

There are 1 GPU(s) available.
We will use the GPU: Tesla T4


device(type='cuda')

## Пример: ruGPT3

Загрузим [ruGPT3](https://huggingface.co/ai-forever/rugpt3large_based_on_gpt2).

ruGPT-3 — семейство русскоязычных языковых моделей на основе архитектуры Transformer, разработанное и развиваемое командой Сбер (SberDevices/Sber AI) как адаптация идей GPT-3 от OpenAI под русский язык.

* Год выпуска: первая версия ruGPT-3 появилась в декабре 2020 г. и развивалась в последующие годы (рост числа параметров и новые версии).

* Число параметров: базовые версии были с разным масштабом (например, от ~760 млн до ~1,3 млрд)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("ai-forever/rugpt3large_based_on_gpt2")

model = AutoModelForCausalLM.from_pretrained("ai-forever/rugpt3large_based_on_gpt2")

config.json:   0%|          | 0.00/622 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/574 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/3.14G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.14G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: ai-forever/rugpt3large_based_on_gpt2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...23}.attn.masked_bias | UNEXPECTED |  | 
transformer.h.{0...23}.attn.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
num_params = sum(p.numel() for p in model.parameters())
print(f"{num_params:,}")
print(f"{num_params/1e6:.1f}M")
print(f"{num_params/1e9:.2f}B")

837,494,784
837.5M
0.84B


In [ ]:
model.cuda()

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 1536)
    (wpe): Embedding(2048, 1536)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-23): 24 x GPT2Block(
        (ln_1): LayerNorm((1536,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=4608, nx=1536)
          (c_proj): Conv1D(nf=1536, nx=1536)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((1536,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=6144, nx=1536)
          (c_proj): Conv1D(nf=1536, nx=6144)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((1536,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=1536, out_features=50257, bias=False)
)

# Zero-shot

Мы используем функцию потерь модели для zero-shot классификации.

Модели на основе GPT используют покомпонентную (per-token) кросс-энтропийную функцию потерь, которая из-за one-hot кодирования токенов сводится к отрицательному логарифму вероятности. **Идея заключается в том, чтобы выбрать целевую метку, связанную с тем промптом, для которого сумма отрицательных логарифмов вероятностей его токенов минимальна.**

Эта идея работает, потому что языковая модель обучена присваивать более высокую вероятность тем продолжениям текста, которые лучше согласуются с контекстом, а класс-метка в zero-shot выступает именно как такое «продолжение».


In [ ]:
import math

def get_loss_num(text):
    # Tokenize the input text and move it to the specified device
    inputs = tokenizer(text, return_tensors="pt").to(device)

    # Shift the inputs to create labels for the next-token prediction task
    labels = inputs["input_ids"].clone() # В задаче language modeling метки — это те же самые токены

    # Move labels to the correct device if you're using GPU
    labels = labels.to(device)

    # Calculate loss
    outputs = model(**inputs, labels=labels)
    loss = outputs.loss
    return loss.item()

### Задача: анализ тональности твитов

Сегодня мы решим задачу анализа тональности. Начнём с нескольких простых (toy) примеров и попробуем придумать такие промпты, которые позволяют различать позитивные и негативные тексты.

**Пример позитивного промпта**

In [ ]:
text = 'жизнь отличная'
get_loss_num('Позитивный твит: ' + text)

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


6.202009201049805

**Negative prompt example**

In [ ]:
get_loss_num('Негативный твит: ' + text)

7.3455810546875

Добавим смайлики

In [ ]:
print(get_loss_num('Позитивный твит: ' + text + ')'))
print(get_loss_num('Негативный твит: ' + text + '('))

5.80208158493042
7.4813666343688965


Теперь мы реализуем функцию, которая выбирает метку, дающую наименьшее значение функции потерь.

In [ ]:
def predict_zero_shot(text, pos = 'Positive text: {})', neg = 'Negative text: {}('):
  pos_loss = get_loss_num(pos.format(text))
  neg_loss = get_loss_num(neg.format(text))
  if pos_loss < neg_loss:
    return 'positive'
  return 'negative'

predict_zero_shot(text)

'positive'

Применим этот подход для классификации тональности твитов

In [ ]:
!wget -O twitter_short.csv https://drive.usercontent.google.com/download?id=17qSrjy5NyknCfhs1kqGwHcHgml9UzpvS&export=download&authuser=0&confirm=t&uuid=cb32846f-bc96-4eb0-9e29-57d27a89e369&at=AN_67v2rr2Fh_KVc0V-EDJQ7bufm:1729946024386

--2026-02-09 10:30:41--  https://drive.usercontent.google.com/download?id=17qSrjy5NyknCfhs1kqGwHcHgml9UzpvS
Resolving drive.usercontent.google.com (drive.usercontent.google.com)... 74.125.137.132, 2607:f8b0:4023:c03::84
Connecting to drive.usercontent.google.com (drive.usercontent.google.com)|74.125.137.132|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 14363 (14K) [application/octet-stream]
Saving to: ‘twitter_short.csv’

twitter_short.csv   100%[===================>]  14.03K  --.-KB/s    in 0s      

2026-02-09 10:30:43 (63.0 MB/s) - ‘twitter_short.csv’ saved [14363/14363]



In [ ]:
import pandas as pd
df = pd.read_csv('twitter_short.csv', index_col = 0)
df.head()

,text,label
0,на работе был полный пиддес :| и так каждое за...,negative
1,"Коллеги сидят рубятся в Urban terror, а я из-з...",negative
2,@elina_4post как говорят обещаного три года жд...,negative
3,"Желаю хорошего полёта и удачной посадки,я буду...",negative
4,"Обновил за каким-то лешим surf, теперь не рабо...",negative


In [ ]:
df.tail()

,text,label
95,"Встречайте, мои супер одногруппницы, будущие и...",positive
96,"все,я вас покидаю,результаты гляну вечером)#би...",positive
97,RT @Dasha_crazy_69: @DashkaTeddy дыы))) но кто...,positive
98,Почти приехали в родное селенье!) @ москва-рига,positive
99,На*уй ваши Канары и Мальдивы ! Тут новая тема ...,positive


In [ ]:
df.label.value_counts()

,count
label,
negative,50
positive,50


In [ ]:
from sklearn.metrics import accuracy_score

df['preds'] = df.text.apply(predict_zero_shot)
accuracy_score(df.label, df.preds)

0.68

In [ ]:
from sklearn.metrics import f1_score

def encode_label(x):
  if x == 'negative':
    return 0
  return 1

f1_score(df.label.apply(encode_label), df.preds.apply(encode_label))

0.7652173913043478

## QWEN2.5

[Qwen2.5](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct) еще одна небольшая LLM.

Qwen 2.5 — это семейство крупномасштабных генеративных языковых моделей от Alibaba Cloud, развитое как следующая итерация после серии Qwen/Qwen 2, с улучшенными способностями к генерации текста, пониманию, кодингу и другим задачам ИИ.

* Официальная дата анонса: 19 сентября 2024 г.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")
model.to(device);

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
num_params = sum(p.numel() for p in model.parameters())
print(f"{num_params:,}")
print(f"{num_params/1e6:.1f}M")
print(f"{num_params/1e9:.2f}B")

1,543,714,304
1543.7M
1.54B


Посмотрим на генерацию

In [ ]:
text = "Продолжи поговорку:\nБез труда"
print(text)

Продолжи поговорку:
Без труда


In [ ]:
tokens = tokenizer(text, add_special_tokens=True, return_tensors="pt").to(device)
tokens

{'input_ids': tensor([[ 53645,   9516,  47081,   1802,   5063,  14497, 125661,  35252,    510,
          60332,  31885,  10813,  19763,  39490]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], device='cuda:0')}

Тестируем:

In [ ]:
outputs = model.generate(**tokens, top_k=1).cpu()
print(tokenizer.batch_decode(outputs)[0])


Продолжи поговорку:
Без труда не выйдешь, ни в чем не поверишь.

Вот продолжение этой


In [ ]:
outputs = model.generate(**tokens, num_beams=4, max_length=30).cpu()
print(tokenizer.batch_decode(outputs)[0])

Продолжи поговорку:
Без труда ничего не добьешься, но без труда ничего и не доб


In [ ]:
outputs = model.generate(**tokens, num_beams=4, num_return_sequences=4, max_length=40).cpu()
print("\n\n\n".join(tokenizer.batch_decode(outputs)))

Продолжи поговорку:
Без труда ничего не добьешься, но без труда ничего и не добьешься.

Исправь ошибки


Продолжи поговорку:
Без труда ничего не добьешься, но без труда ничего и не добьешься.

Вот несколько вариантов


Продолжи поговорку:
Без труда ничего не добьешься, но без труда ничего и не добьешься.

Ваш ответ: Без т


Продолжи поговорку:
Без труда ничего не добьешься, но без труда ничего и не добьешься.

Исправь ошибку


## Системный промпт

**Системный промпт** (или системное сообщение) — это специальная инструкция, передаваемая LLM, которая определяет её поведение, тон, «личность» и ограничения при взаимодействии с пользователем. Он служит базовой настройкой и задаёт ожидания относительно того, как модель должна отвечать на пользовательские запросы на протяжении всей сессии.

Но как это работает? Давайте спросим у [Mistral](https://chat.mistral.ai/), [ChatGPT](https://chatgpt.com) или Gemini! Откройте чат с моделью и введите:

```
Add system prompt in qwen 2.5
```

А теперь давайте добавим системный промпт!


In [ ]:
system_prompt = "Ты — помощник, который генерирует пословицы на русском языке."  # Define your system prompt

prompt = "Продолжи поговорку:\nБез труда"
# Combine system prompt and user prompt into a full prompt
full_prompt = f"{system_prompt}\n\n{prompt}"
# Tokenize the full prompt
tokens = tokenizer(full_prompt, return_tensors="pt").to(device)

# Generate the response using the Qwen-2 model
outputs = model.generate(**tokens, num_beams=4, num_return_sequences=4, max_length=70).cpu()
print("\n\n\n".join([x.split('\n\n')[-1] for x in tokenizer.batch_decode(outputs)]))

Ваш ответ: Без труда ничего не добьешься


Пословица: Без труда ничего не добь


1. Без труда ничего


Следующая поговорка:
Без тру


## ChatTemplate

**ChatTemplate** — это механизм в библиотеке Hugging Face Transformers, который управляет тем, как формируется входной текст (промпт) для чат-ориентированных моделей.

По сути, у каждой чат-модели (например, LLaMA-2-Chat, Qwen-Chat, Mistral-Instruct и других) есть собственный формат диалога — со специальными системными токенами, разделителями между сообщениями пользователя и ассистента, а иногда и дополнительными инструкциями.

Чтобы не писать всё это вручную, в `Transformers` предусмотрена утилита **ChatTemplate**.


**Проще говоря**, это набор правил, который преобразует список сообщений чата (с ролями, такими как система, пользователь, ассистент) в один текстовый промпт, который модель может реально обработать.

### Example

```
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Tell me a joke."}
]
text = chat_template.format(messages)
print(text)

```
Will give us:


```
This template might produce something like:
<|system|>
You are a helpful assistant.
<|user|>
Tell me a joke.
<|assistant|>
That formatted text is what gets tokenized and sent to the model.
```




### Почему это важно

* Разные модели ожидают разные форматы чата (например, OpenAI-style, LLaMA-style, ChatML).
* **ChatTemplate** обеспечивает согласованность токенов между предобучением и инференсом.
* Использование неправильного шаблона может привести к снижению качества ответов или появлению «галлюцинаций».

**⚠ Внимание!** Не стоит слепо доверять рекомендациям LLM по форматам шаблонов чата для конкретных моделей — всегда проверяйте документацию!

**Или просто используйте заранее определённый шаблон!**
Каждая современная чат-модель теперь задаёт свой `chat_template` внутри конфигурации токенизатора:

```python
tokenizer.apply_chat_template(messages, tokenize=False)
```

In [ ]:
prompt = "Продолжи поговорку:\nБез труда"
messages = [
    {"role": "system", "content": "Ты — помощник, который генерирует пословицы на русском языке."},
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
print(text)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=512
)
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
response

<|im_start|>system
Ты — помощник, который генерирует пословицы на русском языке.<|im_end|>
<|im_start|>user
Продолжи поговорку:
Без труда<|im_end|>
<|im_start|>assistant



'Без труда не пришёл и не ушёл.'

## Qwen2.5 для анализа тональности

Посмотрим как эта модель решает задачу анализа тональности.


In [ ]:
text = 'жизнь отличная'
prompt = "Напиши pos в случае если приведенный текст твита позитивный и neg в случае если негативный. Ничего больше не добавляй. Текст твита:\n{}".format(text)
print(prompt)
# Combine system prompt and user prompt into a full prompt
# Tokenize the full prompt
tokens = tokenizer(prompt, return_tensors="pt").to(device)

outputs = model.generate(**tokens, num_beams=2, num_return_sequences=1, max_length=100).cpu()
print(tokenizer.batch_decode(outputs)[0].replace(prompt,''))

Напиши pos в случае если приведенный текст твита позитивный и neg в случае если негативный. Ничего больше не добавляй. Текст твита:
жизнь отличная
, работа хорошая, семья счастливая, друзья хорошие, планы на будущее интересные. Напиши pos в случае если приведенный текст твита позитивный и


Добавим системный промпт.

In [ ]:
system_prompt = "Ты — помощник, который решает задачу sentiment analysis."  # Define your system prompt
text = 'жизнь отличная'

prompt = "Напиши pos в случае если приведенный текст твита позитивный и neg в случае если негативный. Ничего больше не добавляй. Текст твита:\n{}".format(text)
# Combine system prompt and user prompt into a full prompt
full_prompt = f"{system_prompt}\n\n{prompt}"
# Tokenize the full prompt
tokens = tokenizer(full_prompt, return_tensors="pt").to(device)

outputs = model.generate(**tokens, num_beams=2, num_return_sequences=1, max_length=100).cpu()
print(tokenizer.batch_decode(outputs)[0].replace(full_prompt,''))

, всё хорошо

pos, neg

pos, neg

pos, neg

pos, neg

pos, neg

pos, neg

pos, neg

pos


In [ ]:
text = 'жизнь отличная'
prompt = "Напиши pos в случае если приведенный текст твита позитивный и neg в случае если негативный. Ничего больше не добавляй. Текст твита:\n{}".format(text)

messages = [
    {"role": "system", "content": "Ты — помощник, который решает задачу sentiment analysis."},
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
print(text)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=512
)
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
response

<|im_start|>system
Ты — помощник, который решает задачу sentiment analysis.<|im_end|>
<|im_start|>user
Напиши pos в случае если приведенный текст твита позитивный и neg в случае если негативный. Ничего больше не добавляй. Текст твита:
жизнь отличная<|im_end|>
<|im_start|>assistant



'pos'

Модель довольно маленькая, но результат уже хороший.

Давайте потестируем подход с лоссом.

In [ ]:
print(get_loss_num('Это позитивный текст: ' + text))
print(get_loss_num('Это негативный текст: ' + text))

3.2852299213409424
3.3490238189697266


In [ ]:
print(get_loss_num('Это позитивный текст: ' + text + ')'))
print(get_loss_num('Это негативный текст: ' + text + '('))

3.4535913467407227
3.5013387203216553


In [ ]:
from sklearn.metrics import accuracy_score
df['preds_qwen'] = df.text.apply(predict_zero_shot)
accuracy_score(df.label, df.preds_qwen)

0.81

In [ ]:
f1_score(df.label.apply(encode_label), df.preds_qwen.apply(encode_label))

0.7654320987654321